# Notebook to process lysosomal and mitochondrial morphology from cellprofiler features - this time from a merged cell CSV aready created
Inputs Required
 - Database with cellprofiler outputs
   - Mainly use the Per_Cell table for per-cell features
     - Intensity
     - AreaShape features
     - Ratio of organelle area to total cell area
     - Texture features
     - Granularity features
     - Radial intensity distribution about nucleus
   - May additionally want to use mitochdondria or lysosomes.csv if analyzing these specifically 
     - Note that lysosome segmentation is not perfect (but i'm proud of it)
 - Metadata CSV representing 96-well platemap
   - Long-form table containing passage number, staining conditons, treatments, etc
   - Produced automatically from an 8x12 table with the 96Well_PlateMap code
Outputs
- graphs for individual features
- clustering
## Set your paths here

In [ ]:
# Imports
import os
from pathlib import Path
import numpy as np
import pandas as pd

# plotting
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt

%matplotlib inline
import seaborn as sns
import pingouin as pg
import re

from scipy import stats

from plate_information import *
from plate_preprocessing import *
from mitolyso_plot_functions import *
from single_csv_functions import *

## Import the big csv

In [ ]:
# import from a giant csv
# csvpath = "/Volumes/AllieS/Morphology_data/"

# csvpath = "/Volumes/AllieS/2025_OrganelleMorphology_MitoLyso/test_mitoimprov_v5"
csvpath = "/mnt/bigdisk1/AllieSpangaro/Morphology_Replicative_Age_Project/CP_Output/mito_seg_testing/mitoimprovv10"
filename = "Cell.csv"
metadata_path = "/mnt/bigdisk1/AllieSpangaro/Morphology_Replicative_Age_Project/Reworking_Organelle_Segmentation/MitoSegmentation"
metadata_filename = "MitochondriaSegmentationSubsetImages_metaonly_new.csv"
# filename = "total_combined_cell_borders_excluded.csv"

cell_df_mitolyso = pd.read_csv(os.path.join(csvpath, filename))

combined_cell_df_mitolyso = merge_csvs_mitolyso(cell_df_mitolyso, csvpath, "Nuclei.csv")
combined_cell_df_mitolyso = merge_csvs_mitolyso(
    combined_cell_df_mitolyso, csvpath, "RelabeledMito_New.csv"
)
combined_cell_df_mitolyso = merge_csvs_mitolyso(
    combined_cell_df_mitolyso, csvpath, "RelabeledMito_Original.csv"
)
combined_cell_df_mitolyso = merge_csvs_mitolyso(
    combined_cell_df_mitolyso, csvpath, "RelabeledMito_Closed.csv"
)
combined_cell_df_mitolyso = merge_csvs_mitolyso(
    combined_cell_df_mitolyso, csvpath, "RelabeledMito_Puncta_Original.csv"
)

for col in combined_cell_df_mitolyso.columns:
    if "." in col and col.split(".")[0] in combined_cell_df_mitolyso.columns:
        print(f"Dropping dupe column {col}")
        combined_cell_df_mitolyso = combined_cell_df_mitolyso.drop(col, axis=1)
    if col.endswith("_x") and col[:-2] in combined_cell_df_mitolyso.columns:
        print(f"Dropping dupe column {col}")
        combined_cell_df_mitolyso = combined_cell_df_mitolyso.drop(col, axis=1)
    if col.endswith("MitoSkel"):
        print(f"append new to column name for {col}")
        combined_cell_df_mitolyso = combined_cell_df_mitolyso.rename(columns={col: col + "_New"}
        )

# combined_cell_df_mitolyso = pd.merge(combined_cell_df_mitolyso, image_df, left_on=["ImageNumber"], right_on=["ImageNumber"], suffixes=("", "_Image"))
# filter_df = enforce_objects_one_to_one(combined_cell_df_mitolyso)
display(f"cell_df_mitolyso.shape: {cell_df_mitolyso.shape}")
display(f"combined_cell_df_mitolyso.shape: {combined_cell_df_mitolyso.shape}")
# print(cell_df.shape, " ", filter_df.shape)
display(combined_cell_df_mitolyso.head())

In [ ]:
#add metadata columns for merging with metadata csv
def merge_metdata_combined_dataframe(combined_cell_df_mitolyso, metadata_path, metadata_filename, keys = ["Metadata_PlateNumber", "Metadata_RowColField", "Metadata_WellRow", "Metadata_WellColumn", "Metadata_Field"]):
    """Merge the combined cell dataframe with the metadata dataframe based on plate number, well row, well column, and field.

    Args:
        combined_cell_df_mitolyso (pd.DataFrame): The combined cell dataframe.
        metadata_path (str): The path to the directory containing the metadata csv file.
        metadata_filename (str): The name of the metadata csv file.
        keys (list): The list of columns to merge on.

    Returns:
        pd.DataFrame: The combined cell dataframe with the merged metadata.
    """
    combined_cell_df_mitolyso["Metadata_RowColField"] = (
        "r"
        + combined_cell_df_mitolyso["Metadata_WellRow"].astype(str).str.rjust(2, "0")
        + "c"
        + combined_cell_df_mitolyso["Metadata_WellColumn"].astype(str).str.rjust(2, "0")
        + "f"
        + combined_cell_df_mitolyso["Metadata_Field"].astype(str).str.rjust(2, "0")
    )

    combined_cell_df_mitolyso["Metadata_Rep_RowColField"] = (
        "rep"
        + combined_cell_df_mitolyso["Metadata_PlateNumber"].astype(str).str.rjust(2, "0")
        + "_"
        + combined_cell_df_mitolyso["Metadata_RowColField"]
    )

    combined_cell_df_mitolyso["Metadata_PlateNumber"] = combined_cell_df_mitolyso[
        "Metadata_PlateNumber"
    ].astype(str)

    display(f"combined_cell_df_mitolyso.shape before metadata merge: {combined_cell_df_mitolyso.shape}")

    metadata_df = pd.read_csv(os.path.join(metadata_path, metadata_filename))
    metadata_df["Metadata_PlateNumber"] = metadata_df["Metadata_PlateNumber"].astype(str)
    metadata_df["Metadata_RowColField"] = metadata_df["Metadata_RowColField"].astype(str)

    display(metadata_df.tail())
    combined_cell_df_mitolyso = pd.merge(
        combined_cell_df_mitolyso,
        metadata_df,
        on=keys,
        how="left",
    )
    display(
        f"combined_cell_df_mitolyso.shape after metadata merge: {combined_cell_df_mitolyso.shape}"
    )
    return combined_cell_df_mitolyso
combined_cell_df_mitolyso = merge_metdata_combined_dataframe(combined_cell_df_mitolyso, metadata_path, metadata_filename)
display(combined_cell_df_mitolyso.tail())

In [ ]:
# Join another csv with data on the same cells with the ImageJ features
ij_csvpath = "/mnt/bigdisk1/AllieSpangaro/Morphology_Replicative_Age_Project/Reworking_Organelle_Segmentation/MitoSegmentation/MitoSkelMeasurements_V4"
# ij_csv = "mito_masks.csv"
ij_csvs = os.listdir(ij_csvpath)


def get_duplicate_rows(df, subset_cols=None, name="df", keep=False, show=True):
    """
    Return duplicate rows and optionally print a short report.
    Args:
        df (pd.DataFrame): The dataframe to check for duplicates.
        subset_cols (list, optional): The list of columns to check for duplicates. Defaults to None

    subset=[...] checks duplicates only on those columns.
    """
    dup_mask = df.duplicated(subset=subset_cols, keep=keep)
    dup_rows = df.loc[dup_mask].copy()

    if show:
        if subset_cols is None:
            print(f"{name}: {dup_rows.shape[0]} duplicate rows found")
        else:
            print(
                f"{name}: {dup_rows.shape[0]} duplicate rows found for keys {subset_cols}"
            )

    return dup_rows


def merge_ij_skeleton_features_into_combined_dataframe(
    combined_cell_df_mitolyso,
    ij_csvpath,
    ij_csv,
    keys=["Metadata_PlateNumber", "Metadata_RowColField", "ObjectNumber"],
    check_duplicates=True,
    show=False,
):
    """
    Merge the combined cell dataframe with the ImageJ skeleton features dataframe based on plate number
    Args:
        combined_cell_df_mitolyso (pd.DataFrame): The combined cell dataframe.
        ij_csvpath (str): The path to the directory containing the ImageJ skeleton features csv file.
        ij_csv (str): The name of the ImageJ skeleton features csv file.
        keys (list): The list of columns to merge on.
    """
    ij_skeleton_df = pd.read_csv(os.path.join(ij_csvpath, ij_csv))

    csv_name = (
        ij_csv.split(".")[0].strip().title()
    )  # convert to title to match conventions
    csv_suffix = "IJ_Mitochondria_" + csv_name.split("_")[-1].strip()

    # check for dupe keys in the ij_skeleton_df before merging
    if check_duplicates:
        dup_keys = get_duplicate_rows(
            ij_skeleton_df,
            subset_cols=["ImageTitle"] + keys,
            name=f"{ij_csv} merge keys",
            show=show,
        )
        if not dup_keys.empty:
            ij_skeleton_df = ij_skeleton_df.drop_duplicates(
                subset=["ImageTitle"] + keys, keep="first"
            )
            if show:
                display(dup_keys.sort_values(keys))

    for col in ij_skeleton_df.columns:
        new_col = col
        if "+AF8-" in col:
            new_col = col.replace("+AF8-", "_")
        if new_col not in keys + [
            "ImageTitle",
            "Metadata_ThresholdingOP",
        ]:  # rename the column to easily idefntify these features
            new_col = f"{csv_suffix}_{new_col}"
            ij_skeleton_df.rename(columns={col: new_col}, inplace=True)

        ij_skeleton_df["Metadata_PlateNumber"] = ij_skeleton_df[
            "Metadata_PlateNumber"
        ].astype(str)
    if show:
        # display(ij_skeleton_df)
        display(f"shape before merge: {combined_cell_df_mitolyso.shape}")

    combined_cell_df_mitolyso = pd.merge(
        combined_cell_df_mitolyso,
        ij_skeleton_df,
        left_on=keys,
        right_on=keys,
        how="left",
        suffixes=("", f"_{csv_suffix}"),
    )
    if show:
        display(f"shape after merge: {combined_cell_df_mitolyso.shape}")
    return combined_cell_df_mitolyso


def merge_ij_skeleton_features_into_combined_dataframe_from_folder(
    df,
    ij_csvpath,
    ij_csvs=None,
    keys=["Metadata_PlateNumber", "Metadata_RowColField", "ObjectNumber"],
    check_duplicates=True,
    show=False,
):
    """
    Merge the combined cell dataframe with the ImageJ skeleton features dataframes from a folder based on plate number
    Args:
        df (pd.DataFrame): The combined cell dataframe.
        ij_csvpath (str): The path to the directory containing the ImageJ skeleton features csv files.
        ij_csvs (list): The list of ImageJ skeleton features csv files.
        keys (list): The list of columns to merge on.
    """
    combined_cell_df_mitolyso = df.copy()
    if ij_csvs is None:
        ij_csvs = os.listdir(ij_csvpath)
    for item in ij_csvs:
        print(f"Processing {item}...")
        if os.path.isdir(os.path.join(ij_csvpath, item)):
            ij_csv_folder = os.path.join(ij_csvpath, item)
            ij_csv_folder_list = os.listdir(ij_csv_folder)
            for file in ij_csv_folder_list:
                if file.endswith(".csv"):
                    ij_csv = str(file)
                    print(f"Processing {ij_csv} in folder {item}...")
                    combined_cell_df_mitolyso = (
                        merge_ij_skeleton_features_into_combined_dataframe(
                            combined_cell_df_mitolyso,
                            ij_csv_folder,
                            ij_csv,
                            show=show,
                            keys=keys,
                            check_duplicates=check_duplicates,
                        )
                    )
        else:
            combined_cell_df_mitolyso = (
                merge_ij_skeleton_features_into_combined_dataframe(
                    combined_cell_df_mitolyso,
                    ij_csvpath,
                    ij_csv,
                    show=show,
                    keys=keys,
                    check_duplicates=check_duplicates,
                )
            )
    return combined_cell_df_mitolyso

combined_cell_df_mitolyso = merge_ij_skeleton_features_into_combined_dataframe_from_folder(
    combined_cell_df_mitolyso, ij_csvpath, ij_csvs, show=True, check_duplicates=True
)
display(combined_cell_df_mitolyso.head())

## Setup functions


### Filter out the poorly segmented cells and rename the columns so I don't have to change the old code

## Search Column Names

In [ ]:
# colnames
# sns.barplot(filter_df_2, x="AllGroups",y="AreaShape_Area", hue="Plate_Number", palette=colour_dict)
colnames = search_column_name(combined_cell_df_mitolyso, "Mito")
display(combined_cell_df_mitolyso[colnames].head())

### Get original column names 

In [ ]:
def get_unique_cols_to_use(df):
    base_cols = [
        #"FileName_MitoTracker_MAX",
        "Metadata_PlateNumber",
        "Metadata_RowColField",
        "AllGroups",
        "AreaShape_Area",
    ]

    colnames_mitoskel_seeds = search_column_name(df, "MitoSkel_Seeds_ObjectSkeleton")
    colnames_mitoskel_nuc = search_column_name(df, "Nuclei_ObjectSkeleton")
    colnames_mitocount = search_column_name(df, ["Children_Mito", "Count"],inclusive_or=False)
    colnames_mitoarea = search_column_name(df, ["Mito", "AreaShape_Area"],inclusive_or=False)

    colnames_ij = search_column_name(df, "IJ_Mitochondria")

    use_cols = base_cols + colnames_mitoskel_seeds + colnames_mitoskel_nuc+ colnames_ij + colnames_mitocount + colnames_mitoarea
    use_cols_unique =list(dict.fromkeys(use_cols))
    #scrub out any non-numeric columns that we aren't going to use for analysis
    use_cols_unique_copy = use_cols_unique.copy()
    for col in use_cols_unique_copy:
        if col in base_cols:
            continue
        elif "Metadata" in col or "Title" in col or "FileName" in col:
            use_cols_unique.remove(col)
    return use_cols_unique

use_cols_unique = get_unique_cols_to_use(combined_cell_df_mitolyso)

use_cols_unique_copy = use_cols_unique.copy()

display_df = combined_cell_df_mitolyso[use_cols_unique]
display(display_df)


In [ ]:
def make_per_cell_area_column_names(
    df,
    area_col="AreaShape_Area",
    number_of_seeds_col="Children_MitoSkel_Seeds_Count",
    use_cols=use_cols_unique,
    mito_area_cols=[],
    mito_count_cols=[],
    colnames_mitoskel_seeds=[],
    calculate_totals=False,
    base_cols=base_cols,
):
    df = df.copy()
    new_use_cols = use_cols.copy()
    for col in base_cols:
        if col in new_use_cols:
            new_use_cols.remove(col)
    
    new_columns = {}
       
    mito_count_flag = 0
    for col in mito_area_cols:
        #remove the col from new_use_cols so that it doesn't get processed again in the final loop  
        if col in new_use_cols:
            new_use_cols.remove(col)
            
        df[col] = pd.to_numeric(df[col], errors="coerce")
        
        if calculate_totals and "Mean" in col:
            #calculate the total area occupied and then divide by area
            col_without_mean = col.replace("Mean_", "")
            new_columns["Math_Total_" + col_without_mean] = (
                df[col] * df[mito_count_cols[mito_count_flag]]
            )
            new_columns["Per_Area_AreaOccupied_" + col_without_mean] = (
                df["Math_Total_" + col_without_mean] / df[area_col]
            )

        elif "Mean" in col:
            continue
        elif "Total" in col:
            col_without_total = col.replace("Total_", "")
            new_columns["Per_Area_AreaOccupied_" + col_without_total] = df[col] / df[area_col]
        elif "RelabeledMito" in col:
            new_columns["Per_Area_AreaOccupied_" + col] = df[col] / df[area_col]
        else:
            new_columns["Per_Area_" + col] = df[col] / df[area_col]
        
    for col in mito_count_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")
        new_colname = col.replace("Children_", "Number_")
        new_columns["Per_Area_" + new_colname] = df[col] / df[area_col]
        if col in new_use_cols:
            new_use_cols.remove(col)
    # calculate total per cell for mito skel seeded version, then divide by area
    for col in colnames_mitoskel_seeds:
        df[col] = pd.to_numeric(df[col], errors="coerce")
        col_without_mean = col.replace("Mean_", "")
        new_columns["Math_Total_" + col_without_mean] = df[col] * df[number_of_seeds_col]
        new_columns["Per_Area_" + col_without_mean] = new_columns["Math_Total_" + col_without_mean] / df[area_col]
        if col in new_use_cols:
            new_use_cols.remove(col)

    #for everything else in use_cols that makes sense, just divide by area
    for col in new_use_cols:
        if "BranchLength" in col or "Mean" in col or "Median" in col or "Stdev" in col:
            continue
        elif "Metadata" not in col and "Title" not in col:
            df[col] = pd.to_numeric(df[col], errors="coerce")
            new_columns["Per_Area_" + col] = df[col] / df[area_col]
        else:
            continue
        
    new_columns_df = pd.DataFrame(new_columns, index=df.index)
    df = pd.concat([df, new_columns_df], axis=1)   
    return df

combined_cell_df_mitolyso_perarea = make_per_cell_area_column_names(
    combined_cell_df_mitolyso, area_col="AreaShape_Area",number_of_seeds_col="Children_MitoSkel_Seeds_Count", use_cols=use_cols_unique, mito_area_cols=colnames_mitoarea, mito_count_cols=colnames_mitocount, colnames_mitoskel_seeds=colnames_mitoskel_seeds
)

colnames_object_skeleton = search_column_name(
    combined_cell_df_mitolyso_perarea, "ObjectSkeleton"
)
display(combined_cell_df_mitolyso_perarea[colnames_object_skeleton])

In [ ]:
def get_object_skeleton_length_cols(df):
    colnames_object_skeleton_length = search_column_name(
        df, "SkeletonLength"
    )
    for col in colnames_object_skeleton_length.copy():
        if "Mean" in col or "Median" in col or "Stdev" in col or "FromBranches" in col:
            colnames_object_skeleton_length.remove(col)

    print(colnames_object_skeleton_length)
    return colnames_object_skeleton_length


def make_per_skeleton_length_column_names(
    df,
    use_cols,
    skeleton_length_cols,
    base_cols=None,
    sets=[],
    feature_types=[
        "Nuclei_ObjectSkeleton",
        "MitoSkel_Seeds_ObjectSkeleton",
        "IJ_Mitochondria",
    ],
):
    df = df.copy()
    new_use_cols = use_cols.copy()
    if base_cols is None:
        base_cols = [
            "Metadata_PlateNumber",
            "Metadata_RowColField",
            "AllGroups",
            "AreaShape_Area",
        ]   
    # take out things that don't make sense
    for col in use_cols:
        if (
            col in base_cols
            or col in skeleton_length_cols
            or "Metadata" in col
            or "Title" in col
            or "Mean" in col
            or "Median" in col
            or "Stdev" in col
            or "Footprint" in col
            or "Per_Area" in col
        ):
            new_use_cols.remove(col)

    new_columns = {}

    for col in new_use_cols:
        # check if the column is one of the feature types we want to process
        this_feature_type = None
        for feature_type in feature_types:
            if feature_type in col:
                this_feature_type = feature_type
                break

        df[col] = pd.to_numeric(df[col], errors="coerce")

        for length_col in skeleton_length_cols:
            # only divide by the skeleton length column that matches the feature type of the current column
            if this_feature_type and this_feature_type not in length_col:
                continue
            elif this_feature_type == "IJ_Mitochondria":
                subtypes = ["_Max", "_Total"]
                if any(subtype in col for subtype in subtypes) and any(
                    (subtype in col and subtype not in length_col)
                    or (subtype not in col and subtype in length_col)
                    for subtype in subtypes
                ):
                    continue

            if sets and any(set_name in length_col for set_name in sets):
                for set_name in sets:
                    if set_name in length_col and set_name in col:
                        print(
                            f"Calculating Per_SkeletonLength for {col} using {length_col}"
                        )
                        new_columns["Per_SkeletonLength_" + col] = df[col] / df[length_col]
                        break
            else:
                new_columns["Per_SkeletonLength_" + col] = df[col] / df[length_col]

    new_columns_df = pd.DataFrame(new_columns, index=df.index)
    df = pd.concat([df, new_columns_df], axis=1)    
    return df

colnames_object_skeleton_length = get_object_skeleton_length_cols(
    combined_cell_df_mitolyso_perarea
)

combined_cell_df_mitolyso_perarea_perskel = make_per_skeleton_length_column_names(
    combined_cell_df_mitolyso_perarea,
    use_cols=colnames_mitoskel_nuc + colnames_mitoskel_seeds + colnames_ij,
    skeleton_length_cols=colnames_object_skeleton_length,
    sets=["New", "Closed", "Original", "Puncta"],
    feature_types=[
        "Nuclei_ObjectSkeleton",
        "MitoSkel_Seeds_ObjectSkeleton",
        "IJ_Mitochondria",
    ],
)
display(combined_cell_df_mitolyso_perarea_perskel.tail())

In [ ]:
colnames_per_area = search_column_name(combined_cell_df_mitolyso_perarea_perskel, "Per_Area")
colnames_per_skeletonlength = search_column_name(combined_cell_df_mitolyso_perarea_perskel, "Per_SkeletonLength")
#colnames_totals = search_column_name(combined_cell_df_mitolyso_perarea_perskel, "Total")

use_cols_new = use_cols_unique + colnames_per_area + colnames_per_skeletonlength
use_cols_new_unique =list(dict.fromkeys(use_cols_new))


display_df = combined_cell_df_mitolyso_perarea_perskel[use_cols_new_unique]
display(display_df)
display_df_pivot = display_df.pivot_table(
    index=["AllGroups"],
    values=use_cols_new_unique[3:],
    aggfunc="mean",
)

display_df_pivot_plates = display_df.pivot_table(
    index=["Metadata_PlateNumber", "AllGroups"],
    values=use_cols_new_unique[3:],
    #columns=["AllGroups"],
    aggfunc="mean",
)


display(display_df_pivot)
display(display_df_pivot_plates)


In [ ]:
summary_stats = display_df.describe()
outpath = "segmentation_test"
display_df.to_csv(os.path.join(outpath, "processed_cell_data.csv"), index=False) 
display_df_pivot.to_csv(os.path.join(outpath, "processed_cell_data_pivot.csv"), index=True)
summary_stats.to_csv(os.path.join(outpath, "processed_cell_data_summary.csv"), index=True)

In [ ]:
feature_cols = use_cols_new_unique[3:]
colour_dict = get_hard_code_plate_colours(
    combined_cell_df_mitolyso_perarea_perskel, ["1", "2", "3", "4", "5", "6", "7"], "Metadata_PlateNumber"
)
def plot_boxplots_with_swarm(display_df, display_df_pivot_plates, feature_cols, colour_dict=colour_dict, size=(6, 10)):
    outpath = "segmentation_test/plots"
    os.makedirs(outpath, exist_ok=True)
    for col in feature_cols:
        plt.figure(figsize=size)
        sns.set_style("whitegrid")
        sns.set_context("notebook")
        sns.boxplot(display_df, x="AllGroups", y=col, palette="pastel" , fill=True, hue="AllGroups", legend=False)
        sns.swarmplot(display_df_pivot_plates, x="AllGroups", y=col, palette=colour_dict,hue="Metadata_PlateNumber", dodge=False, size = 9, legend=False, linewidth=1, edgecolor="k" )
        sns.despine()
        plt.ylim(0, np.percentile(display_df[col].dropna(), 99)*1.1)  
        plt.tight_layout()
        plt.savefig(os.path.join(outpath, f"{col}_boxplot.png"))
        plt.show()
        
# def plot_boxplots_with_swarm_pairs(
#     display_df,
#     display_df_pivot_plates,
#     feature_cols,
#     colour_dict=colour_dict,
#     size=(6, 10),
# ):
#     outpath = "segmentation_test/pairedplots"
#     os.makedirs(outpath, exist_ok=True)
#     for col in feature_cols:
#         fig, axes = plt.subplots(1, 2, figsize=(figsize[0] * 2, figsize[1]), sharey=True)

#         for ax, suffix in zip(axes, ["1", "2"]):
#             item = rows[suffix]
#         sns.set_style("whitegrid")
#         sns.set_context("notebook")
#         ax = plt.gca()
#         ax = sns.boxplot(
#             display_df,
#             x="AllGroups",
#             y=col,
#             palette="pastel",
#             fill=True,
#             hue="AllGroups",
#             legend=False,
#         )
#         sns.swarmplot(
#             display_df_pivot_plates,
#             x="AllGroups",
#             y=col,
#             palette=colour_dict,
#             hue="Metadata_PlateNumber",
#             dodge=False,
#             size=9,
#             legend=False,
#             linewidth=1,
#             edgecolor="k",
#         )
#         sns.despine()
#         plt.ylim(0, np.percentile(display_df[col].dropna(), 99) * 1.1)
#         plt.tight_layout()
#         plt.savefig(os.path.join(outpath, f"{col}_boxplot.png"))
#         plt.show()

plot_boxplots_with_swarm(display_df, display_df_pivot_plates, feature_cols, colour_dict, size=(6, 10))

In [ ]:
def plot_boxplots_with_swarm_tukey(
    display_df,
    display_df_pivot_plates,
    feature_cols,
    colour_dict=colour_dict,
    size=(6, 10),
    order=None,
    group_col="AllGroups",
    plate_col="Metadata_PlateNumber",
):
    outpath = "segmentation_test/plots_tukey"
    os.makedirs(outpath, exist_ok=True)
    from statannotations.Annotator import Annotator

    if order is None:
        order = display_df[group_col].dropna().unique().tolist()

    for col in feature_cols:
        plt.figure(figsize=size)
        sns.set_style("whitegrid")
        sns.set_context("talk")
        ax = sns.boxplot(
            data=display_df,
            x=group_col,
            y=col,
            palette="pastel",
            fill=True,
            hue=group_col,
            legend=False,
        )
        sns.swarmplot(
            data=display_df_pivot_plates,
            x=group_col,
            y=col,
            palette=colour_dict,
            hue=plate_col,
            dodge=False,
            size=9,
            legend=False,
            linewidth=1,
            edgecolor="k",
        )

        pairs = getpairs(display_df, group_col, order=order)
        pairs, p_values = pvalues_anova_and_tukeyhsd_posthoc(
            display_df,
            display_df_pivot_plates,
            x_value=group_col,
            y_value=col,
            plate_number_col=plate_col,
            desired_pairs=pairs,
            order=order,
        )

        if pairs:
            annotator = Annotator(
                ax=ax,
                pairs=list(pairs),
                data=display_df,
                plot="boxplot",
                x=group_col,
                y=col,
                order=order,
            )
            annotator.reset_configuration()
            annotator.configure(
                text_format="full",
                test_short_name="tukey_v3",
                pvalue_format_string="{:.4f}",
                fontsize="small",
                loc="inside",
                hide_non_significant=True,
                color="black",
                verbose=2,
                line_height=0.01,
                text_offset=0.5,
                show_test_name=False,
            )
            annotator.set_pvalues_and_annotate(p_values)

        sns.despine()
        if display_df[col].dropna().shape[0] > 0:
            upper = np.nanpercentile(display_df[col].dropna(), 99) * 1.2
            if np.isfinite(upper):
                plt.ylim(0, upper)
        plt.tight_layout()
        plt.savefig(os.path.join(outpath, f"{col}_boxplot_tukey.png"))
        plt.show()
        
plot_boxplots_with_swarm_tukey(
    display_df, display_df_pivot_plates, feature_cols, colour_dict, size=(6, 12)
)


# Define the cell features


## Feature lists here:

In [ ]:
feature_dicts = [mito_features, lyso_features, nuc_features, cell_features]
feature_names = [
    "Mitochondria Features",
    "Lysosome Features",
    "Nucleus Features",
    "Cell Features",
]

# Define the output file path
output_file_path = "allfeatures_file.md"

# Open the file in write mode
with open(output_file_path, "w") as file:
    for feature_name, feature_dict in zip(feature_names, feature_dicts):
        file.write(f"# {feature_name}\n\n")
        for feature_type, features in feature_dict.items():
            file.write(f"## {feature_type.capitalize()}\n")
            for feature in features:
                file.write(f'"{feature}",\n')
            file.write("\n")

print(f"List has been written to {output_file_path}")


## Normalize features to control (Passage 6-8)

In [ ]:
# updated/faster version that doesn't use apply and is much faster for large dataframes
all_valid_features = []
for feat_dict in feature_dicts:
    all_cols = [col for cols in feat_dict.values() for col in cols]
    for col in set(all_cols):
        if pd.api.types.is_numeric_dtype(extrafeatures_filtered_cell_df_mitolyso[col]):
            all_valid_features.append(col)
valid_feature_cols = list(
    dict.fromkeys(all_valid_features)
)  # remove duplicates while preserving order
group_col = "AgeGroup"
control_value = 0
norm_combined_cell_df_mitolyso = normalize_quantities_to_control_group_average(
    extrafeatures_filtered_cell_df_mitolyso.copy(),
    valid_feature_cols,
    group_col,
    control_value,
    overwrite=True,
    drop_avg_cols=True,
)
# display(
#     norm_combined_cell_df_mitolyso[
#         ["AllGroups", "Plate_Number", "Metadata_Well"]
#         + valid_feature_cols
#     ]
# )
# norm_combined_cell_df_mitolyso.boxplot(column="Mean_Lysosomes_Distance_Centroid_Cell", by="AllGroups")
# norm_combined_cell_df_mitolyso.groupby(["Plate_Number", "AllGroups"])[
#     "Mean_Lysosomes_Distance_Centroid_Cell"
# ].describe()

In [ ]:
def allgroups_sort_key(value):
    """Custom sort key function for 'AllGroups' column."""
    import re

    match = re.match(r"P(\d+)", value)
    if match:
        first_number = match.group(1)
        if first_number.isdigit():
            return int(first_number)
        else:
            return 99999
    return 99999


print(allgroups_sort_key("P6-10"))

## Helper functions for plot building

In [ ]:
def annotate_pairs_with_calculated_pvalues_test(
    ax,
    data,
    pivot_data,
    x_value,
    y_value,
    plate_col_name="Plate_Name",
    test_name="tukey",
    pairs=None,
    order=None,
    plot_type="violinplot",
    show_test_name=False,
    p_correction="fdr_bh",
    annotation_location="inside",
):
    """Add statistical annotations to a plot using a multiple comparisons test in the statsmodels, scipy, or scikit-posthocs modules.
    see https://statannotations.readthedocs.io/en/latest/custom-test.html for more examples
    Also see https://www.graphpad.com/guides/prism/latest/statistics/stat_summary_of_multiple_comparison.htm for a list of posthoc tests and when to use them

    Args:
        ax (Matplotlib Axes Object): the axis of the graph to annoate
        data (DataFrame): dataframe from a grouped feature df containing the groups aggregated by plate to analyze in "tidy" format
        pivot_data (DataFrame): the grouped feature df in matrix format
        x_value (str): independent variable on x axis
        y_value (str): dependent variable on y axis
        plate_col_name (str, optional): the col containing the experimental plate. Defaults to "Plate_Name".
        test_name (str, optional): the statistical test to perform. Accepts values of "tukey", "anova", or "tukey_v2", for ANOVA with Tukey's HSD, "tukey_v3" for ANOVA with Tukey HSD with Tukey-Kramer correction, "games-howell" or "games" for ANOVA with Games-Howell posthoc, "rmanova" for repeated-measures ANOVA using Welch's ttest with the specified p-value correction, "kruskal" or "dunn" for classic nonparametric multiple comparisons with Dunn's postc, "conover" for kruskal with Conover's posthoc, "nemenyi" for kruskal (or friedman) with Nemenyi's posthoc for repeated measures, "pairwise_ttest" for corrected ttests, "pairwise_mwu" for nonparametic multiple comparisons. Defaults to "tukey".
        pairs (list of str, optional): The pairs of x_value for the comparisons. Defaults to None, is automatically calculated otherwise based on the getpairs() function.
        order (listlike, optional): _description_. Defaults to None.
        plot (str, optional): the type of plot to annotate. Defaults to "violinplot".
        p_correction (str, optional): the p-value correction to use if applicable. Deaults to Benjamini/Hochberg "fdr_bh" (non-negative) method ; graphpad reccomneds as its less hemmoraging to your power. Also accepts "holm", "sidak", "bonferroni", "holm-sidak" and Benjamini/Yekutieli "fdr-by" for negative values. See https://scikit-posthocs.readthedocs.io/en/latest/generated/scikit_posthocs.posthoc_mannwhitney.html for other options

    Returns:
        _type_: _description_
    """
    from statannotations.Annotator import Annotator

    if pairs is None:
        pairs = getpairs(data, x_value, order=order)

    # nonparametric tests
    if test_name in ["kruskal", "dunn", "kruskal-wallis"]:
        used_pairs, p_values = kruskal_with_dunn_posthoc(
            data,
            x_value=x_value,
            y_value=y_value,
            order=order,
            desired_pairs=pairs,
            p_correction=p_correction,
            display_results=True,
        )
    elif test_name in ["drubin", "drubin_posthoc"]:
        used_pairs, p_values = kruskal_with_drubin_posthoc(
            data,
            x_value=x_value,
            y_value=y_value,
            order=order,
            desired_pairs=pairs,
            p_correction=p_correction,
            display_results=True,
        )
    elif test_name in ["conover", "con", "kruskal-conover"]:
        used_pairs, p_values = kruskal_with_conover_posthoc(
            data,
            x_value=x_value,
            y_value=y_value,
            order=order,
            desired_pairs=pairs,
            p_correction=p_correction,
            display_results=True,
        )
    elif test_name in ["nemenyi", "kruskal-nemenyi"]:
        used_pairs, p_values = kruskal_with_nemenyi_posthoc(
            data,
            x_value=x_value,
            y_value=y_value,
            order=order,
            desired_pairs=pairs,
            display_results=True,
            p_correction=p_correction,
        )
    # parametric tests
    elif test_name in ["anova", "tukey", "tukeyhsd"]:
        # perform anova and tukey's post-hoc test
        used_pairs, p_values = pvalues_anova_and_tukeyhsd_posthoc(
            data, pivot_data, x_value, y_value, order=order, desired_pairs=pairs
        )
    elif test_name in ["games", "games-howell"]:
        used_pairs, p_values = pvalues_anova_with_games_howell_pingouin(
            data,
            pivot_data,
            x_value,
            y_value,
            order=order,
            desired_pairs=pairs,
            display=True,
        )
    elif test_name in ["tahmane", "tahmane-t2"]:
        used_pairs, p_values = anova_with_tahmane_posthoc(
            data,
            x_value,
            y_value,
            order=order,
            desired_pairs=pairs,
            display_results=True,
        )
    elif test_name in ["tukey_v2", "tukey_posthocs"]:
        used_pairs, p_values = anova_with_tukey_posthoc(
            data,
            x_value=x_value,
            y_value=y_value,
            plate_number_col=plate_col_name,
            order=order,
            desired_pairs=pairs,
            display_results=True,
        )
    elif test_name in ["tukey_v3", "tukey_pingouin"]:
        used_pairs, p_values = pvalues_anova_with_tukey_pingouin(
            data,
            pivot_data,
            x_value=x_value,
            y_value=y_value,
            plate_number_col=plate_col_name,
            order=order,
            desired_pairs=pairs,
            display=True,
        )
    elif test_name in ["pairwise_ttest" or "multiple_ttest"]:
        used_pairs, p_values = pvalues_anova_with_pairwise_tests_pingouin(
            data,
            x_value=x_value,
            y_value=y_value,
            order=order,
            desired_pairs=pairs,
            pval_correction=p_correction,
            parametric=True,
            display=True,
        )
    elif test_name in [
        "pairwise_mwu"
        or "multiple_mwu"
        or "pairwise_mannwhitney"
        or "multiple_mannwhitney"
        or "pairwise_wilcoxon"
        or "multiple_wilcoxon"
    ]:
        used_pairs, p_values = pvalues_anova_with_pairwise_tests_pingouin(
            data,
            x_value=x_value,
            y_value=y_value,
            order=order,
            desired_pairs=pairs,
            pval_correction=p_correction,
            parametric=False,
            display=True,
        )
    elif test_name in ["ttest_posthoc", "welch's_posthoc", "tt_posthoc"]:
        used_pairs, p_values = anova_with_corr_ttest_posthoc(
            data,
            x_value=x_value,
            y_value=y_value,
            order=order,
            desired_pairs=pairs,
            p_corr=p_correction,
            display_results=True,
        )
    else:
        raise ValueError(
            f"Test name '{test_name}' is invalid. Use 'tukey', 'anova', 'kruskal', 'games-howell', 'drubin', 'tukey_v2', or 'dunn'."
        )
    if used_pairs is None or len(used_pairs) == 0:
        print(f"No significant pairs found for the {test_name} test.")
        return ax
    else:
        annotator = Annotator(
            ax=ax,
            pairs=list(used_pairs),
            data=data,
            plot=plot_type,
            x=x_value,
            y=y_value,
            order=order,
        )
        annotator.reset_configuration()
        # see https://raw.githubusercontent.com/trevismd/statannotations/3f020ae631ca88a091b6ee3e9a9fd32158920879/usage/example_tuning_y_offsets_w_arguments.png
        annotator.configure(
            text_format="full",
            test_short_name=test_name,
            pvalue_format_string="{:.4f}",
            fontsize="small",
            # pvalue_format = [[1e-5, "1e-5"], [1e-4, "1e-4"], [1e-3, "0.001"], [1e-2, "0.01"], [5e-2, "0.05"]],
            loc=annotation_location,
            hide_non_significant=True,
            color="black",
            verbose=2,
            line_height=0.01,
            text_offset=0.8,
            # line_offset_to_group=0.01,
            # line_offset=0.01,
            show_test_name=show_test_name,
        )
        annotator.set_pvalues_and_annotate(p_values)
        return ax


def super_splitviolinplot_helper_singleplot(
    data_df,
    group_avg_df,
    ax,
    x_value,
    y_value,
    title,
    plate_col_name,
    pairs=None,
    order=None,
    annotate=False,
    test=None,
    shapiro=True,
    show_test_on_plot=False,
    pallete=None,
    p_correction="bonf",
    annotation_location="inside",
):
    group_avg_df = group_avg_df.copy()
    if pairs is None:
        pairs = getpairs(data_df, x_value, order=order)

    if pallete is None:
        pallete = get_hard_code_plate_colours(group_avg_df)
    print(pairs)
    sns.violinplot(
        data=data_df,
        x=x_value,
        y=y_value,  # hue=x_value,
        # palette="Set2",
        split=True,  # using split violin plots - only one side, basically looks like a histogram
        inner="quart",
        color="gainsboro",
        dodge=False,
        # fill = True
        width=1,
        linewidth=1.5,
        order=order,
        ax=ax,
        cut=0.5,
    )
    # add in the colour scheme
    unique_plates = group_avg_df[plate_col_name].unique()
    group_avg_df[plate_col_name] = pd.Categorical(
        group_avg_df[plate_col_name], categories=unique_plates
    )
    sns.swarmplot(
        data=group_avg_df,
        x=x_value,
        y=y_value,
        hue=plate_col_name,
        order=order,
        palette=pallete,
        size=10,
        edgecolor="k",
        linewidth=1,
        dodge=False,
        ax=ax,
    )
    # draw a boxplot to show the mean line
    sns.boxplot(
        data=group_avg_df,
        x=x_value,
        y=y_value,
        showmeans=True,
        meanline=True,
        meanprops={"color": "dimgray", "ls": "-", "lw": 4},
        medianprops={"visible": False},
        whiskerprops={"visible": False},
        zorder=2,
        showfliers=False,
        showbox=False,
        showcaps=False,
        ax=ax,
    )
    ax.set_title(title)

    # use pivot table to get the average values for each group
    if annotate and test is not None:
        group_avg_pivot_table = average_groups_pivot(
            group_avg_df, x_value, y_value, plate_col_name
        )
        try:
            ax = annotate_pairs_with_calculated_pvalues_test(
                ax,
                group_avg_df,
                group_avg_pivot_table,
                x_value,
                y_value,
                plate_col_name=plate_col_name,
                test_name=test,
                order=order,
                plot_type="violinplot",
                show_test_name=show_test_on_plot,
                p_correction=p_correction,
                annotation_location=annotation_location,
            )
        except Exception as e:
            print(f"Error annotating with statistical test: {e}")
            # ax = annotate_with_anova_tukey(ax, pairs, group_avg_df_pivot, x_value, y_value, plate_col_name=plate_col_name, order=order, plot="violinplot")
        # elif test == "kruskal":
        #     ax = annotate_with_kruskal(
        #         ax,
        #         pairs,
        #         group_avg_pivot_table,
        #         x_value,
        #         y_value,
        #         order=order,
        #         plate_col_name=plate_col_name,
        #         plot="violinplot",
        #     )
        if shapiro:
            ax = annotate_legend_with_shapiro(ax, group_avg_df, plate_col_name)

    return ax


def single_feature_super_splitviolinplot(
    data_df,
    x_value="AllGroups",
    y_value="Cell_AreaShape_Area",
    plate_col_name="Plate_Number",
    out_dir=Path(""),
    xtitle=None,
    ytitle=None,
    order=None,
    legend=True,
    annotate=False,
    test=None,
    show_hist=False,
    remove_outliers=False,
    rm_outliers_method="mad",
    ylim=None,
    reps_to_exclude=[],
    shapiro=True,
    show=True,
    context="talk",
    font_scale=1.2,
    figsize=(8, 6),
    truncate_outliers=False,
    norm=False,
    pallete=None,
    p_correction="bonf",
    annotation_location="inside",
):
    """Make a superplot to do multiple comparisons for a feature between different conditions
    Args:
        data_df_1 (_type_): _description_
        group_avg_df_1 (_type_): _description_
        data_df_2 (_type_): _description_
        group_avg_df_2 (_type_): _description_
        x_value (str, optional): _description_. Defaults to "AllGroups".
        y_value (str, optional): _description_. Defaults to "Cell_AreaShape_Area".
        plate_col_name (str, optional): _description_. Defaults to "Plate_Number".
        csv_dir (str, optional): _description_. Defaults to "".
        xtitle (_type_, optional): _description_. Defaults to None.
        ytitle (_type_, optional): _description_. Defaults to None.
    """
    import matplotlib.lines as mlines
    from statannotations.Annotator import Annotator
    from statannotations.stats.StatTest import StatTest
    from pathlib import Path

    if order == None:
        order = get_all_group_order()
    pairs = getpairs(data_df, x_value, order=order)
    print(pairs)
    if truncate_outliers:
        try:
            bottom_fence = None  # np.percentile(data_df[y_value], 0.000001)
            if norm:
                top_fence = data_df[y_value].mean() + 10 * data_df[y_value].std()
            else:
                top_fence = np.percentile(data_df[y_value], 99.9)
                # handle errors where the data is very skewed and the percentile is inf or nan
                if top_fence == 0 or np.isnan(top_fence) or np.isinf(top_fence):
                    top_fence = None
            axlim = (bottom_fence, top_fence)
        except ValueError as e:
            print(e)
            top_fence = None
            bottom_fence = None
            axlim = (None, None)
    else:
        axlim = (None, None)

    df_sorted = data_df.sort_values(
        by=[x_value], key=lambda x: x.map(passage_groups_sort_key)
    ).reset_index(drop=True)
    if show_hist:
        hist = sns.kdeplot(
            df_sorted, x=y_value, hue=plate_col_name, palette=pallete, multiple="layer"
        )
        plt.xlim(axlim)
        plt.savefig(f"{Path(out_dir, f'{y_value}_{plate_col_name}_histogram')}.png")
        plt.show()
        plt.close()
        hist2 = seaborn_ridgeplot(
            df_sorted,
            value_col=y_value,
            group_col=x_value,
            palette="Set2",
            save=True,
            out_dir=out_dir,
            show_percentiles=True,
        )
        plt.close()
    fig, ax = plt.subplots(figsize=figsize)
    sns.set_theme(style="ticks", context=context, font_scale=font_scale)

    feature_df = df_sorted[[x_value, y_value, plate_col_name]].copy()
    if reps_to_exclude:
        feature_df = feature_df[~feature_df[plate_col_name].isin(reps_to_exclude)]
        print(f"removing plates: {reps_to_exclude}")

    # remove outluers
    if remove_outliers is True:
        if rm_outliers_method == "mad":
            feature_df = flag_outliers_by_group_mad(feature_df, x_value, y_value)
            feature_df = feature_df[~feature_df["Outlier"]]
        elif rm_outliers_method == "gesd":
            feature_df = flag_outliers_by_group_gesd(
                feature_df, x_value, y_value, noutliers=50
            )
            feature_df = feature_df[~feature_df["Outlier_GESD"]]
        elif rm_outliers_method == "gesd_2":
            print(feature_df.shape)
            feature_df = remove_outliers_by_group_gesd(
                feature_df, x_value, y_value, noutliers=500
            )
            print(feature_df.shape)
        elif rm_outliers_method == "tietjen":
            feature_df = remove_outliers_by_group_tietjen(
                feature_df, x_value, y_value, noutliers=50
            )
        elif rm_outliers_method == "iqr":
            feature_df = remove_outliers_iqr(feature_df)
        else:
            ValueError(f"{rm_outliers_method} is not a valid outlier removal method")
        # display(feature_df)

    # group by plate and condition
    group_avg_df = feature_df.groupby([x_value, plate_col_name], as_index=False).mean()
    # sort the group avg df by the "AllGroups" order
    group_avg_df_sorted = group_avg_df.sort_values(
        by=[x_value], key=lambda x: x.map(allgroups_sort_key)
    ).reset_index(drop=True)

    ax = super_splitviolinplot_helper_singleplot(
        feature_df,
        group_avg_df_sorted,
        ax,
        x_value,
        y_value,
        title=" ",
        plate_col_name=plate_col_name,
        pairs=pairs,
        order=order,
        annotate=annotate,
        test=test,
        shapiro=False,
        pallete=pallete,
        p_correction=p_correction,
        annotation_location=annotation_location,
    )

    if legend:
        ax.legend(
            bbox_to_anchor=(1.02, 1),
            loc="upper left",
            frameon=True,
            title=plate_col_name,
        )
        if shapiro:
            group_avg_df_shapiro = apply_shapiro_wilk_test_to_df(
                group_avg_df_sorted,
                feature_meas=y_value,
                plate_col_name="Plate_Number",
                alpha=0.05,
            )
            # display(group_avg_df_shapiro)
            ax = annotate_legend_with_shapiro(ax, group_avg_df_shapiro, plate_col_name)
    else:
        ax.legend_.remove()
    if ytitle is not None:
        ax.set_ylabel(ytitle)
    else:
        ax.set_ylabel(y_value.replace("_", " "))
    if xtitle is not None:
        ax.set_xlabel(xtitle)

    # Set the ylim and don't throw an inf
    if ylim is None:
        ylim = axlim
    if ylim[0] is not None and ylim[1] is not None:
        if not (
            np.isnan(ylim[0])
            or np.isnan(ylim[1])
            or np.isinf(ylim[0])
            or np.isinf(ylim[1])
        ):
            ax.set_ylim(ylim)
    plt.tight_layout()
    sns.despine(trim=True)
    plt.savefig(os.path.join(out_dir, f"{y_value}_{test}.png"))
    if show:
        plt.show()

## Make a plot for a single feature

In [ ]:
order = get_all_group_order()
feature_meas = "AreaShape_Area"  # "AreaShape_Area"#Mean_Lysosomes_Distance_Centroid_Nuclei_PerCell_Area" #Mean_Lysosomes_DiameterRatio_PerCell"
ylabel = None  # "Mitochondrial Density Per Cell (relative to youngest passage)"#None#"Mitochondria per cell"
xlabel = "Age Groups"
group = "AllGroups"

pairs = getpairs(combined_cell_df_mitolyso, group, order)

# this_df = extrafeatures_filtered_cell_df_mitolyso.copy()
this_df = norm_combined_cell_df_mitolyso.copy()

data_df = this_df  # >3500]   #Potentially use a theshold - this is the trough of the nuc size peak at 0
# this_df = extrafeatures_filtered_cell_df_mitolyso.copy()

# Map each plate to a color and hard code that shit
colour_dict = get_hard_code_plate_colours(data_df)
pallete = colour_dict  # "pastel"

remove_outliers = False
reps_to_exclude = []
plot_dir = "plots/notnorm"
os.makedirs(plot_dir, exist_ok=True)

figsize = (13, 12)

single_feature_super_splitviolinplot(
    data_df,
    x_value=group,
    y_value=feature_meas,
    plate_col_name="Plate_Number",
    xtitle=xlabel,
    ytitle=ylabel,
    out_dir=plot_dir,
    annotate=True,
    order=order,
    test="tukey_v3",
    reps_to_exclude=reps_to_exclude,
    show_hist=False,  # True,
    remove_outliers=remove_outliers,
    rm_outliers_method="gesd_2",
    truncate_outliers=True,
    legend=True,
    context="talk",
    font_scale=0.8,
    figsize=figsize,
    pallete=pallete,
    p_correction="fdr_bh",
    annotation_location="inside",
    # truncate_outliers=True
)

# NOTE: R5 has smallest cells in p23-25, which is also highest mito density

## Make multiple plots as defined

In [ ]:
# take feature : label pairs to be used for plots from csv
features_df = pd.read_csv("CP_features_for_plots.csv")
display(features_df[features_df["feature"] == "AreaShape_Area"])

data_df = extrafeatures_filtered_cell_df_mitolyso.copy()
# Use in plotting loops with guaranteed alignment:


def make_feature_plots_from_csv(
    data_df,
    features_df,
    xlabel="Age Groups",
    group="AllGroups",
    analysis_mode="Plates",
    norm=False,
    order=None,
    remove_outliers=False,
    reps_to_exclude=[],
    figsize=(14, 18),
    truncate_outliers=True,
    rm_outliers_method="gesd_2",
    annotate_pval=True,
    test="tukey_v3",
    font_scale=0.8,
    annotation_location="inside",
    show_legend=False,
):
    features_for_plots = features_df.to_dict("records")

    feature_df_cols = [item["feature"] for item in features_for_plots]
    feature_labels = [item["label"] for item in features_for_plots]

    if order is None:
        order = get_all_group_order()
    pairs = getpairs(data_df, group, order)

    # Set stats variables and color palette based on analysis mode
    if analysis_mode == "Lineages":
        stat_grouping = "Lineage"
        plot_dir = Path("plots/lineages")
        show_legend = True
        colour_dict = get_hard_code_lineage_colours(data_df)
    elif analysis_mode == "Plates":
        stat_grouping = "Plate_Number"
        colour_dict = get_hard_code_plate_colours(data_df)
        if norm:
            data_df = norm_combined_cell_df_mitolyso.copy()
            feature_labels = [
                f"{name} (normalized to youngest group)" for name in feature_labels
            ]
            plot_dir = Path("plots/norm")
        else:
            data_df.copy()
            data_df = data_df[data_df["MitoEnds_NumberBranchEnds_PerCell_Area"] > 0]
            plot_dir = Path("plots/notnorm")
            
    else:
        raise ValueError(
            f"Invalid analysis mode: {analysis_mode}. Use 'Lineages' or 'Plates'."
        )
    pallete = colour_dict  # "pastel"

    os.makedirs(plot_dir, exist_ok=True)
    for i, feature in enumerate(feature_df_cols):
        ylabel = feature_labels[i]

        single_feature_super_splitviolinplot(
            data_df,
            x_value=group,
            y_value=feature,
            plate_col_name=stat_grouping,
            xtitle=xlabel,
            ytitle=ylabel,
            out_dir=plot_dir,
            annotate=annotate_pval,
            order=order,
            test=test,
            reps_to_exclude=reps_to_exclude,
            show_hist=False,  # True,
            remove_outliers=remove_outliers,
            rm_outliers_method=rm_outliers_method,
            truncate_outliers=truncate_outliers,
            legend=show_legend,
            context="talk",
            figsize=figsize,
            pallete=pallete,
            p_correction="fdr_bh",
            shapiro=False,
            font_scale=font_scale,
            annotation_location=annotation_location,
        )


make_feature_plots_from_csv(
    data_df,
    features_df,
    norm=True,
    analysis_mode="Plates",
    remove_outliers=False,
    reps_to_exclude=[],
    figsize=(13, 12),
    annotate_pval=True,
    test="tukey_v3",
    font_scale=0.8,
    annotation_location="inside",
)